# Perform data preprocessing with pycytominer on single cell features

Note: Single cell is represented by only nuclei compartment features which was used to extract features across all three channels.

## Import libraries

In [1]:
import pathlib

import pandas as pd
from pycytominer import normalize, feature_select

## Set paths and variables

In [2]:
# Path to dir with converted profiles per plate (each plate as a folder)
converted_dir = pathlib.Path("./data/converted_profiles")

# Path to dir with `qc.parquet` with metadata for which nuclei failed qc
qc_dir = pathlib.Path("./data/single_cell_qc")

# path for plate map directory
platemap_dir = pathlib.Path("../0.download_data/metadata/platemaps")

# Output dir for the files to be saved to
output_dir = pathlib.Path("./data/single_cell_profiles")
output_dir.mkdir(exist_ok=True, parents=True)

# operations to perform for feature selection
feature_select_ops = [
    "variance_threshold",
    "correlation_threshold",
    "blocklist",
]

## Perform preprocessing on single cell features

In [3]:
profile_path = pathlib.Path(f"{converted_dir}/u2os_per_nuclei.parquet")

print("Performing pycytominer pipeline for dataset")

output_normalized_file = pathlib.Path(
    f"{output_dir}/u2os_per_nuclei_sc_normalized.parquet"
)
output_feature_select_file = pathlib.Path(
    f"{output_dir}/u2os_per_nuclei_sc_feature_selected.parquet"
)

# Load data
profile_df = pd.read_parquet(profile_path)

qc_file = qc_dir / "qc.parquet"
qc_df = pd.read_parquet(qc_file)

# ---- Define join columns ----
join_cols = [
    "Image_Metadata_Plate",
    "Image_Metadata_Well",
    "Image_Metadata_Position",
    "Image_Metadata_Best_Z",
    "Nuclei_Location_Center_X",
    "Nuclei_Location_Center_Y",
]

# ---- Select all Metadata_cqc columns ----
qc_cqc_cols = [col for col in qc_df.columns if col.startswith("Metadata_cqc")]

qc_subset = qc_df[join_cols + qc_cqc_cols]

# ---- Merge QC into profile ----
profile_df = profile_df.merge(qc_subset, on=join_cols, how="left")

# ---- Drop rows where any QC column indicates failure (True) ----
qc_fail_cols = [col for col in qc_subset.columns if col.startswith("Metadata_cqc")]
annotated_df = profile_df[~profile_df[qc_fail_cols].any(axis=1)]
print(
    f"After QC filtering, {len(annotated_df)} nuclei remain out of {len(profile_df)} total nuclei."
)

# Rename columns using the rename() function
column_name_mapping = {
    "Image_Metadata_Plate": "Metadata_Plate",
    "Image_Metadata_Well": "Metadata_Well",
    "Image_Metadata_Position": "Metadata_Position",
    "Image_Metadata_Best_Z": "Metadata_Best_Z",
    "Image_Count_Nuclei": "Metadata_Nuclei_Site_Count",
    "Nuclei_Location_Center_X": "Metadata_Nuclei_Location_Center_X",
    "Nuclei_Location_Center_Y": "Metadata_Nuclei_Location_Center_Y",
    "Nuclei_AreaShape_BoundingBoxMaximum_X": "Metadata_Nuclei_AreaShape_BoundingBoxMaximum_X",
    "Nuclei_AreaShape_BoundingBoxMaximum_Y": "Metadata_Nuclei_AreaShape_BoundingBoxMaximum_Y",
    "Nuclei_AreaShape_BoundingBoxMinimum_X": "Metadata_Nuclei_AreaShape_BoundingBoxMinimum_X",
    "Nuclei_AreaShape_BoundingBoxMinimum_Y": "Metadata_Nuclei_AreaShape_BoundingBoxMinimum_Y",
}

annotated_df.rename(columns=column_name_mapping, inplace=True)

# Step 2: Normalization
normalized_df = normalize(
    profiles=annotated_df,
    method="mad_robustize",
    output_file=output_normalized_file,
    output_type="parquet",
)

# Step 3: Feature selection
feature_select(
    output_normalized_file,
    operation=feature_select_ops,
    output_file=output_feature_select_file,
    na_cutoff=0,
    output_type="parquet",
    blocklist_file="./blocklist_features.txt",
)
print(
    "Annotation, normalization, and feature selection have been performed for dataset"
)

Performing pycytominer pipeline for dataset
After QC filtering, 8874 nuclei remain out of 9142 total nuclei.
Annotation, normalization, and feature selection have been performed for dataset


## Check example output file to confirm that the process worked

In [4]:
# Check output file
test_df = pd.read_parquet(output_feature_select_file)

print(test_df.shape)
test_df.head(5)

(8874, 291)


,Metadata_ImageNumber,Metadata_Plate,Metadata_Position,Metadata_Best_Z,Metadata_Well,Metadata_Nuclei_Site_Count,Metadata_Nuclei_AreaShape_BoundingBoxMaximum_X,Metadata_Nuclei_AreaShape_BoundingBoxMaximum_Y,Metadata_Nuclei_AreaShape_BoundingBoxMinimum_X,Metadata_Nuclei_AreaShape_BoundingBoxMinimum_Y,...,Nuclei_Texture_InfoMeas1_SON_3_01_256,Nuclei_Texture_InfoMeas1_SRRM2_3_01_256,Nuclei_Texture_InfoMeas2_DAPI_3_01_256,Nuclei_Texture_InfoMeas2_DAPI_3_03_256,Nuclei_Texture_InfoMeas2_SRRM2_3_02_256,Nuclei_Texture_InverseDifferenceMoment_DAPI_3_00_256,Nuclei_Texture_InverseDifferenceMoment_DAPI_3_02_256,Nuclei_Texture_SumVariance_DAPI_3_01_256,Nuclei_Texture_SumVariance_SON_3_01_256,Nuclei_Texture_SumVariance_SRRM2_3_00_256
0,56,Rep1,pos11,bestZ08,well4,91,2714.0,225.0,2506.0,106.0,...,1.795736,1.943433,-0.711279,-0.700575,-3.328748,1.948966,1.564543,-0.933724,-1.038068,-1.168952
1,76,Rep2,pos01,bestZ08,well2,106,2785.0,208.0,2663.0,62.0,...,-0.607479,-0.471681,0.002011,-0.471643,0.658964,-1.739082,-0.939509,0.608319,0.771924,0.541633
2,37,Rep1,pos07,bestZ08,well3,90,2916.0,295.0,2728.0,137.0,...,0.989238,1.510113,-2.049936,-1.797395,-1.209154,1.202147,0.954939,-0.959003,-0.295226,-0.445915
3,38,Rep1,pos08,bestZ08,well3,52,3522.0,242.0,3391.0,49.0,...,0.759008,0.451430,-0.028979,-0.035819,-0.274509,-0.511915,0.205687,0.160582,-0.020575,-0.206161
4,17,Rep1,pos02,bestZ08,well2,24,1152.0,327.0,974.0,198.0,...,-0.194308,-0.491279,-0.345905,-0.359903,-0.000173,0.950187,0.472903,-0.555017,0.150269,-0.192656
